## ⚠️ Notebook status (updated dataset)

These notebooks were originally created as experiments (02–14). Their historical runs used **deprecated / non-integral data**.

As of **May 24, 2026**, the integral dataset is:
`dataset/raw/handover_dataset.csv`

Important dataset property:
- `optimal_cell_idx_in_k` is **constant 0** in the integral dataset because neighbor lists are score-sorted with the optimal cell at index 0.
- Any pipeline that trains directly on `optimal_cell_idx_in_k` without **neighbor-axis shuffling** will learn order leakage and produce misleading accuracy.

Recommended usage now:
- Treat notebooks 02–14 as **robustness / stability / scalability** harnesses.
- For a leakage-safe pointer target, reuse `src/preprocess.py` (`dataset/processed/*.npz`) or `src/production/temporal_deepset_data.py`.

See production docs:
- `docs/production_constraints.md`
- `docs/robustness_scalability_plan.md`
- `docs/explainability_and_finetuning.md`


In [4]:
# --- Multi-horizon window controls (NEW DATASET) ---
# Each row in the dataset corresponds to one measurement interval.
MEASUREMENT_INTERVAL_MS = 50

# History window length (timesteps). Example: 25 → 1.25 s history @ 50 ms.
WIN_T = 25

# Multi-horizon label generation. Example: 5 → predict up to 250 ms ahead.
FUTURE_H = 5

# Optional lead time before horizon starts (in timesteps).
LEAD_L = 0

# For notebooks that are NOT multi-output, choose which horizon to train on (1..FUTURE_H).
TARGET_H_IDX = 1

# Label source:
# - "optimal": train against oracle best cell (optimal_cell_id)
# - "target" : train against executed target (target_cell_id)
LABEL_MODE = "optimal"

# Feature toggles for cache builder:
# - include_scores=True adds nb_scores to per-cell features
# - include_global=True adds [speed, cos(dir), sin(dir), cell_load] to each cell feature vector
INCLUDE_SCORES = True
INCLUDE_GLOBAL = False

# Set True to force rebuilding dataset/mh_cache for new window/horizon settings
FORCE_REBUILD = False

# Use the leakage-safe multi-horizon cache loader.
USE_MH_CACHE = True

print(
    f"[window] dt={MEASUREMENT_INTERVAL_MS}ms  "
    f"T={WIN_T} ({WIN_T*MEASUREMENT_INTERVAL_MS}ms)  "
    f"H={FUTURE_H} ({FUTURE_H*MEASUREMENT_INTERVAL_MS}ms)  "
    f"lead={LEAD_L}  target_h={TARGET_H_IDX}"
)


[window] dt=50ms  T=25 (1250ms)  H=5 (250ms)  lead=0  target_h=1


In [5]:
# ─── SECTION 4: UNIFIED DATA PIPELINE (Multi-Horizon & Strictly Balanced) ───
# Inspired by 01_temporal_deepset.ipynb, but generalized for Multi-Horizon
# Provides X, M, y (multi-horizon), r (regression), and y_bin (binary HO).
SEED = 42
from pathlib import Path
import logging
try: _r = _ROOT
except NameError: _r = Path("../../").resolve()
try: log.info
except NameError: log = logging.getLogger("dummy"); log.setLevel(logging.INFO)
import re
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
def parse_nb_array(s, max_len=10, fill=0.0):
    if pd.isna(s): return [fill] * max_len
    cleaned = re.sub(r'[\[\]]', '', str(s)).strip()
    parts = re.split(r'[,;]', cleaned)
    vals = []
    for p in parts[:max_len]:
        p = p.strip()
        if p == '' or p.lower() in ('nan', 'none'): vals.append(fill)
        else:
            try: vals.append(float(p))
            except: vals.append(fill)
    vals += [fill] * (max_len - len(vals))
    return vals
def parse_nb_ids(s, max_len=10):
    if pd.isna(s): return [0] * max_len
    cleaned = re.sub(r'[\[\]]', '', str(s)).strip()
    parts = re.split(r'[,;]', cleaned)
    ids = []
    for p in parts[:max_len]:
        p = p.strip()
        try: ids.append(int(float(p)))
        except: ids.append(0)
    ids += [0] * (max_len - len(ids))
    return ids
def load_and_create_mh_datasets(root_dir, win_t=25, future_h=5, lead_l=0, k_cells=10):
    raw_path = root_dir / "dataset" / "raw" / "handover_dataset.csv"
    log.info(f"Loading raw data from {raw_path}...")
    df = pd.read_csv(raw_path, low_memory=False)
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed")
    
    df["nb_ids"]   = df["nb_cell_ids"].apply(parse_nb_ids)
    df["nb_rsrps"] = df["nb_rsrps"].apply(parse_nb_array)
    df["nb_sinrs"] = df["nb_sinrs"].apply(parse_nb_array)
    df["nb_loads"] = df["nb_loads"].apply(parse_nb_array)
    
    df.sort_values(["ue_id", "timestamp"], inplace=True)
    
    all_X, all_M, all_y, all_r, groups = [], [], [], [], []
    rng_shuf = np.random.default_rng(SEED)
    
    log.info(f"Building MH sequences (T={win_t}, H={future_h}, L={lead_l}) with mandatory shuffling...")
    for ue_id, grp in df.groupby("ue_id", sort=False):
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        n_rows = len(grp)
        if n_rows < win_t + lead_l + future_h: continue
        
        cell_feat = np.zeros((n_rows, k_cells, 4), dtype=np.float32)
        for i in range(n_rows):
            rs, sn, ld = grp.at[i, "nb_rsrps"], grp.at[i, "nb_sinrs"], grp.at[i, "nb_loads"]
            # 4th feature is placeholder for score if needed, or 0
            for k in range(k_cells):
                cell_feat[i, k, 0] = rs[k]
                cell_feat[i, k, 1] = sn[k]
                cell_feat[i, k, 2] = ld[k]
                cell_feat[i, k, 3] = 0.0 # score placeholder
                
        opt_ids = grp["optimal_cell_id"].values
        ue_nb_ids = grp["nb_ids"].values
        ue_srv_ids = grp["serving_cell_id"].values
        opt_rsrp = grp["optimal_cell_rsrp"].values
        
        for t in range(win_t, n_rows - (lead_l + future_h) + 1):
            X_w = cell_feat[t-win_t : t] 
            p = rng_shuf.permutation(k_cells)
            X_w_shuf = X_w[:, p, :].transpose(1, 0, 2)
            M_w = (X_w_shuf[:, -1, 0] != 0.0).astype(np.float32)
            
            yh = np.zeros((future_h,), dtype=np.int32)
            rh = np.zeros((future_h,), dtype=np.float32)
            
            current_nb_ids = ue_nb_ids[t-1]
            
            for step in range(future_h):
                f_idx = t + lead_l + step
                chosen_id = int(opt_ids[f_idx])
                rh[step] = float(opt_rsrp[f_idx])
                try:
                    orig = current_nb_ids.index(chosen_id)
                    yh[step] = int(np.where(p == orig)[0][0])
                except ValueError:
                    srv = int(ue_srv_ids[f_idx])
                    try:
                        orig = current_nb_ids.index(srv)
                        yh[step] = int(np.where(p == orig)[0][0])
                    except ValueError:
                        yh[step] = 0
            
            all_X.append(X_w_shuf)
            all_M.append(M_w)
            all_y.append(yh)
            all_r.append(rh)
            groups.append(ue_id)
    X = np.array(all_X)
    M = np.array(all_M)
    y = np.array(all_y)
    r = np.array(all_r, dtype=np.float32)
    groups = np.array(groups)
    
    # Split
    ue_list = np.unique(groups)
    np.random.default_rng(SEED).shuffle(ue_list)
    n_te, n_va = int(len(ue_list)*0.15), int(len(ue_list)*0.15)
    ue_te, ue_va = set(ue_list[:n_te]), set(ue_list[n_te:n_te+n_va])
    
    idx_tr = np.where([u not in ue_te and u not in ue_va for u in groups])[0]
    idx_va = np.where([u in ue_va for u in groups])[0]
    idx_te = np.where([u in ue_te for u in groups])[0]
    
    # Scale Features
    scaler_x = StandardScaler()
    scaler_x.fit(X[idx_tr][M[idx_tr]==1].reshape(-1, 4))
    X_n = X.copy()
    for i in range(len(X_n)):
        vk = np.where(M[i] == 1.0)[0]
        if len(vk) > 0:
            X_n[i, vk] = scaler_x.transform(X[i, vk].reshape(-1, 4)).reshape(-1, win_t, 4)
            
    # Scale Regression
    scaler_r = StandardScaler()
    scaler_r.fit(r[idx_tr].reshape(-1, 1))
    r_n = scaler_r.transform(r.reshape(-1, 1)).reshape(-1, future_h)
    
    return {
        "train": {"X": X_n[idx_tr], "M": M[idx_tr], "y": y[idx_tr], "r": r_n[idx_tr]},
        "val":   {"X": X_n[idx_va], "M": M[idx_va], "y": y[idx_va], "r": r_n[idx_va]},
        "test":  {"X": X_n[idx_te], "M": M[idx_te], "y": y[idx_te], "r": r_n[idx_te]},
        "scalers": {"x": scaler_x, "r": scaler_r}
    }
# Execute unified pipeline
try: _wt = WIN_T
except: _wt = HP.get("OBS_STEPS", 25)
try: _fh = FUTURE_H
except: _fh = 5
try: _ll = LEAD_L
except: _ll = 0
_data = load_and_create_mh_datasets(_r, win_t=_wt, future_h=_fh, lead_l=_ll, k_cells=10)
# Map to canonical variables
X_tr, M_tr, y_tr_mh, r_tr = _data["train"]["X"], _data["train"]["M"], _data["train"]["y"], _data["train"]["r"]
X_va, M_va, y_va_mh, r_va = _data["val"]["X"], _data["val"]["M"], _data["val"]["y"], _data["val"]["r"]
X_te, M_te, y_te_mh, r_te = _data["test"]["X"], _data["test"]["M"], _data["test"]["y"], _data["test"]["r"]
scaler = _data["scalers"]["x"]
scaler_r = _data["scalers"]["r"]
# For Multi-Horizon prediction, preserve all horizons
y_tr = y_tr_mh.astype("int32")
y_va = y_va_mh.astype("int32")
y_te = y_te_mh.astype("int32")
# If notebook expects binary HO labels
y_bin_tr = (y_tr > 0).astype(np.float32)
y_bin_va = (y_va > 0).astype(np.float32)
y_bin_te = (y_te > 0).astype(np.float32)
# Ensure N_FEATS matches our 4-feature output (which might have 3 valid ones)
try:
    if "N_FEATS" in HP:
        HP["N_FEATS"] = 3  # nb_score removed
except NameError:
    pass
log.info("Unified MH Datasets Ready! X_tr: %s, y_tr: %s", X_tr.shape, y_tr.shape)


10:50:20 │ INFO     │ Loading raw data from /home/wassimmchichi/Downloads/Handover_projects/dataset/raw/handover_dataset.csv...


10:50:25 │ INFO     │ Building MH sequences (T=25, H=5, L=0) with mandatory shuffling...
10:50:47 │ INFO     │ Unified MH Datasets Ready! X_tr: (57120, 10, 25, 4), y_tr: (57120, 5)


# 03 · Experiment 3 — Multi-Task Learning Set Transformer
## Objective: Break the 57% Top-1 Ceiling

### Strategy: Delta & Margin Features + Dual-Head MTL Architecture

```
WHY THE 57% CEILING EXISTS
─────────────────────────────────────────────────────────────────────────────
Previous models saw:  [rsrp_t, sinr_t, load_t, score_t]  — static snapshots
                       ↑ blind to whether rsrp is rising or falling

MTL Solution:
  1. Feature Engineering  → [rsrp, sinr, load, score, Δrsrp, margin]
                             F: 4  →  6
     • Δrsrp  = rsrp_t − rsrp_{t-1}     fading velocity (is signal improving?)
     • margin = rsrp_cell − rsrp_serving  competitive gap (how close is the rival?)

  2. Multi-Task Heads
     Head A (Focal Loss)  → argmax cell for handover  (classification)
     Head B (MSE Loss)    → expected RSRP of target at t+1  (regression)
     ↑ Regression forces the shared encoder to learn signal physics,
       regularising the representation and closing the accuracy gap.
```

### Timing Contract
```
Observation (5 s = 25 steps)  │  Gap (1 s = 5 steps)  │  Target (1 s = 5 steps)
    features [t : t+25]        │    (latency dead-zone)  │  label [t+30 : t+35]
```

### Total Loss
```
L_total = 1.0 × L_focal  +  0.5 × L_mse
```

### Directory Layout (notebook at `notebooks/modeling/`)
```
../../
├── dataset/mtl_cache/        ← augmented cache (built by Cell 3)
├── models/                   ← best_mtl_transformer.keras
├── metrics/                  ← plots, metadata, training_log.csv
├── tb_logs/mtl_transformer/  ← TensorBoard events
└── mlflow/mlruns/            ← MLflow (EC2 via SSH tunnel or env-var)
```

## Section 1 · Environment, Paths, GPU, MLflow

In [6]:
# ─── Section 1 · Environment ─────────────────────────────────────────────────
#
# Notebook lives at notebooks/modeling/03_mtl_transformer.ipynb
# All I/O is anchored at the project root via Path("../../").resolve()
# This works regardless of where `jupyter lab` is launched from.

import os, sys, warnings, json, pickle, logging, datetime, gc, re
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing  import Tuple, List, Dict, Optional

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from sklearn.preprocessing      import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics            import (classification_report,
                                        confusion_matrix,
                                        top_k_accuracy_score,
                                        mean_absolute_error)
sns.set_theme(style="whitegrid", font_scale=1.05)

# ── Root-anchored paths ───────────────────────────────────────────────────────
_ROOT = Path("../../").resolve()

PATHS = dict(
    raw_csv   = _ROOT / "dataset" / "raw"         / "handover_dataset.csv",
    mtl_cache = _ROOT / "dataset" / "mtl_cache",
    models    = _ROOT / "models",
    metrics   = _ROOT / "metrics"/ "mtl_transformer",
    tb_logs   = _ROOT / "tb_logs" / "mtl_transformer",
    mlruns    = _ROOT / "mlflow"  / "mlruns",
)
for p in (PATHS["mtl_cache"], PATHS["models"],
          PATHS["metrics"],   PATHS["tb_logs"],
          PATHS["mlruns"]):
    os.makedirs(str(p), exist_ok=True)

# ── Dual logging ──────────────────────────────────────────────────────────────
_log_file = PATHS["metrics"] / "mtl_training.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s │ %(levelname)-8s │ %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout),
              logging.FileHandler(str(_log_file), mode="w")]
)
log = logging.getLogger("mtl")
log.info("Root : %s", _ROOT)
for k,v in PATHS.items(): log.info("  %-12s → %s", k, v)

# ── GPU ───────────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices("GPU")
log.info("GPUs: %d", len(gpus))
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
    log.info("  %s — memory_growth=True", g.name)
if not gpus:
    log.warning("No GPU — running on CPU.")

# RTX 2060 = Turing sm_75 → FP16 Tensor Cores
policy = mixed_precision.Policy("mixed_float16")
mixed_precision.set_global_policy(policy)
log.info("Mixed precision: compute=%s  vars=%s",
         policy.compute_dtype, policy.variable_dtype)

# ── MLflow — EC2 via SSH tunnel (or env-var override) ─────────────────────────
# To use tunnel:   ssh -N -L 5000:localhost:5000 -i key.pem ec2-user@<EC2_IP>
# To skip tunnel:  export MLFLOW_TRACKING_URI=http://<EC2_PUBLIC_IP>:5000
_MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
try:
    import mlflow, mlflow.tensorflow, requests
    try:
        _r = requests.get(f"{_MLFLOW_URI}/health", timeout=3)
        MLFLOW_OK = _r.status_code == 200
    except Exception:
        MLFLOW_OK = False
    if MLFLOW_OK:
        mlflow.set_tracking_uri(_MLFLOW_URI)
        mlflow.set_experiment("handover_MTL_Transformer")
        log.info("MLflow → %s", _MLFLOW_URI)
    else:
        log.warning("MLflow unreachable at %s (tunnel open?)", _MLFLOW_URI)
except ImportError:
    MLFLOW_OK = False
    log.warning("mlflow not installed.")

SEED = 48
tf.random.set_seed(SEED); np.random.seed(SEED)
log.info("TF %s | NumPy %s", tf.__version__, np.__version__)

10:50:47 │ INFO     │ Root : /home/wassimmchichi/Downloads/Handover_projects
10:50:47 │ INFO     │   raw_csv      → /home/wassimmchichi/Downloads/Handover_projects/dataset/raw/handover_dataset.csv
10:50:47 │ INFO     │   mtl_cache    → /home/wassimmchichi/Downloads/Handover_projects/dataset/mtl_cache
10:50:47 │ INFO     │   models       → /home/wassimmchichi/Downloads/Handover_projects/models
10:50:47 │ INFO     │   metrics      → /home/wassimmchichi/Downloads/Handover_projects/metrics/mtl_transformer
10:50:47 │ INFO     │   tb_logs      → /home/wassimmchichi/Downloads/Handover_projects/tb_logs/mtl_transformer
10:50:47 │ INFO     │   mlruns       → /home/wassimmchichi/Downloads/Handover_projects/mlflow/mlruns
10:50:47 │ INFO     │ GPUs: 1
10:50:47 │ INFO     │   /physical_device:GPU:0 — memory_growth=True
10:50:47 │ INFO     │ Mixed precision: compute=float16  vars=float32


2026/05/30 10:50:47 INFO mlflow.tracking.fluent: Experiment with name 'handover_MTL_Transformer' does not exist. Creating a new experiment.


10:50:47 │ INFO     │ MLflow → http://127.0.0.1:5000
10:50:47 │ INFO     │ TF 2.15.1 | NumPy 1.26.4


## Section 2 · Hyperparameters

In [7]:
# ─── Section 2 · Hyperparameters ─────────────────────────────────────────────

HP = dict(
    # ── Data ──────────────────────────────────────────────────────────────────
    MAX_CELLS   = 10,
    OBS_STEPS   = 25,
    N_FEATS     = 4,     # ← CHANGED: 4 base + 2 augmented (delta, margin)
    LAT_STEPS   = 5,
    TGT_STEPS   = 5,

    # ── Loss weights ──────────────────────────────────────────────────────────
    # L_total = LAMBDA_CLS × L_focal  +  LAMBDA_REG × L_mse
    LAMBDA_CLS   = 1.0,
    LAMBDA_REG   = 0.5,
    FOCAL_GAMMA  = 2.0,
    FOCAL_ALPHA  = 0.25,

    # ── Architecture ──────────────────────────────────────────────────────────
    LSTM_UNITS   = 64,
    PHI_DIM      = 64,
    N_HEADS      = 4,
    MHA_KEY_DIM  = 16,
    FF_DIM       = 128,
    N_ST_BLOCKS  = 2,
    DROPOUT      = 0.20,

    # ── Training ──────────────────────────────────────────────────────────────
    BATCH_SIZE   = 128,
    EPOCHS       = 60,
    LR_INIT      = 1e-3,
    LR_WARMUP_EP = 4,
    LR_DECAY_EP  = 20,
)

ALL_LABELS = list(range(HP["MAX_CELLS"]))   # [0..9] for top_k_accuracy_score
F_NAMES = ["nb_rsrp", "nb_sinr", "nb_load"]  # nb_score removed — leakage
CKPT_PATH  = str(PATHS["models"] / "best_mtl_transformer.keras")
FINAL_PATH = str(PATHS["models"] / "mtl_transformer_final.keras")

log.info("HP loaded. N_FEATS=%d (4 base + 2 augmented)", HP["N_FEATS"])
log.info("Loss: %.1f × Focal + %.1f × MSE", HP["LAMBDA_CLS"], HP["LAMBDA_REG"])

# --- Override window controls (injected) ---
try:
    HP["OBS_STEPS"] = int(WIN_T)
    if "PRED_STEPS" in HP: HP["PRED_STEPS"] = int(FUTURE_H)
    if "TGT_STEPS"  in HP: HP["TGT_STEPS"]  = int(FUTURE_H)
    if "LAT_STEPS"  in HP: HP["LAT_STEPS"]  = int(LEAD_L)
except Exception as _e:
    print('HP override skipped:', _e)


10:50:47 │ INFO     │ HP loaded. N_FEATS=4 (4 base + 2 augmented)
10:50:47 │ INFO     │ Loss: 1.0 × Focal + 0.5 × MSE


## Section 3 · Feature Augmentation — Delta & Margin Strategy

### Why these two features break the ceiling

| Feature | Formula | Physical meaning |
|---|---|---|
| **RSRP Δ** | `rsrp_t − rsrp_{t−1}` | Fading velocity — is this cell's signal strengthening or weakening? A cell at −60 dBm and rising is better than one at −55 dBm and falling. |
| **RSRP Margin** | `rsrp_cell − rsrp_serving` | Competitive gap — how far ahead of the serving cell is this candidate? The handover trigger condition (A3 event) fires when margin > hysteresis. |

Both features are **dimensionless relative quantities** computed from the existing
`nb_rsrp` column — zero new columns needed from the CSV.

The preprocessing also extracts `target_rsrp` (mean `optimal_cell_rsrp` over the
5-row target window) as the regression ground truth for Head B.

In [8]:
# ─── Section 3a · Preprocessing helpers ──────────────────────────────────────

_BRACKET = re.compile(r'[\[\]]')

def parse_floats(raw: str, k: int = HP["MAX_CELLS"]) -> np.ndarray:
    s   = _BRACKET.sub("", str(raw)).strip()
    out = np.full(k, np.nan, dtype=np.float32)
    for i, p in enumerate(s.split(";")[:k]):
        try: out[i] = float(p.strip())
        except ValueError: pass
    return out

def parse_ints(raw: str, k: int = HP["MAX_CELLS"]) -> List[int]:
    s   = _BRACKET.sub("", str(raw)).strip()
    out = [0] * k
    for i, p in enumerate(s.split(";")[:k]):
        try: out[i] = int(float(p.strip()))
        except ValueError: pass
    return out

def step_features(row: pd.Series) -> Tuple[np.ndarray, np.ndarray, List[int]]:
    """Extract base (MAX_CELLS, 3) feature matrix + mask from one row."""
    rsrps  = parse_floats(row["nb_rsrps"])
    sinrs  = parse_floats(row["nb_sinrs"])
    loads  = parse_floats(row["nb_loads"])
    # scores removed — nb_scores encodes optimal_cell_score criterion (leakage)
    feat   = np.stack([rsrps, sinrs, loads], axis=1).astype(np.float32)
    mask   = (~np.isnan(feat[:, 0])).astype(np.float32)
    np.nan_to_num(feat, nan=0.0, copy=False)
    return feat, mask, parse_ints(row["nb_cell_ids"])

def derive_label(tgt_rows: pd.DataFrame, anchor_ids: List[int]) -> int:
    acc: Dict[int, List[float]] = {}
    for _, row in tgt_rows.iterrows():
        cid = int(row["optimal_cell_id"])
        acc.setdefault(cid, []).append(float(row["optimal_cell_score"]))
    best = max(acc, key=lambda c: float(np.mean(acc[c])))
    return anchor_ids.index(best) if best in anchor_ids else 0

def derive_target_rsrp(tgt_rows: pd.DataFrame) -> float:
    """
    Mean optimal_cell_rsrp over the 5 target rows.
    This is Head B's regression ground truth: the expected signal
    strength of the best cell 1 s into the future.
    """
    return float(tgt_rows["optimal_cell_rsrp"].mean())


def augment_features(X_base: np.ndarray) -> np.ndarray:
    """
    Inject two trajectory features into a (N, MAX_CELLS, W, 4) tensor.

    Feature 5 — RSRP Δ (delta):
        rsrp_t − rsrp_{t-1}  along the W time axis.
        Step 0 uses prepend=step_0 → delta[0]=0.0 (no prior context).
        Positive values = signal strengthening (good candidate).
        Negative values = fading (avoid this cell).

    Feature 6 — RSRP Margin:
        rsrp[cell_i] − rsrp[cell_0=serving]  across the C cell axis.
        Cell 0 is always the serving cell → its margin is always 0.
        Positive margin → candidate stronger than current serving cell.

    Returns: (N, MAX_CELLS, W, 6) float32
    """
    # delta: shape (N, C, W)
    delta  = np.diff(X_base[..., 0], axis=2,
                     prepend=X_base[:, :, :1, 0])       # (N, C, W)
    # margin: shape (N, C, W)
    serving = X_base[:, 0:1, :, 0]                      # (N, 1, W)
    margin  = X_base[:, :, :, 0] - serving              # (N, C, W)

    return np.concatenate([
        X_base,
        delta[..., np.newaxis],
        margin[..., np.newaxis],
    ], axis=-1).astype(np.float32)


log.info("Feature augmentation helpers defined.")
log.info("  augment_features: (N,C,W,4) → (N,C,W,6)")
log.info("  F layout: [nb_rsrp, nb_sinr, nb_load, nb_score, rsrp_Δ, rsrp_margin]")

10:50:47 │ INFO     │ Feature augmentation helpers defined.
10:50:47 │ INFO     │   augment_features: (N,C,W,4) → (N,C,W,6)
10:50:47 │ INFO     │   F layout: [nb_rsrp, nb_sinr, nb_load, nb_score, rsrp_Δ, rsrp_margin]


## Section 4 · tf.data Pipeline — Dual-Output Dataset

The dataset produces **two labels per sample**:
- `y_cls` — one-hot cell index `(MAX_CELLS=10,)` for Head A (Focal Loss)
- `y_reg` — scalar target RSRP `()` for Head B (MSE Loss)

Keras's `model.fit()` accepts a dict of outputs as targets, matching
the output layer names `cls_output` and `reg_output`.

In [9]:
# ─── Section 4 · Data pipeline ───────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE
cw_vals      = compute_class_weight("balanced",
                                     classes=np.unique(y_tr), y=y_tr.flatten())
CLASS_WEIGHT = {int(c): float(w) for c, w in enumerate(cw_vals)}
log.info("Class weights: %s", {k:round(v,3) for k,v in CLASS_WEIGHT.items()})
def make_ds(X: np.ndarray, M: np.ndarray,
            y: np.ndarray, r: np.ndarray,
            sw: np.ndarray = None,  # ← New optional sample weights
            shuffle: bool = False) -> tf.data.Dataset:
    
    y_oh = tf.one_hot(y, depth=HP["MAX_CELLS"]).numpy().astype(np.float32)
    
    inputs  = {"cells": X, "mask": M}
    targets = {"cls_output": y_oh, "reg_output": r.astype(np.float32)}
    
    # If sample weights are provided, add them as the 3rd element
    with tf.device('/CPU:0'):
        if sw is not None:
            weights = {"cls_output": sw.astype(np.float32)}
            ds = tf.data.Dataset.from_tensor_slices((inputs, targets, weights))
        else:
            ds = tf.data.Dataset.from_tensor_slices((inputs, targets))
        
    if shuffle:
        ds = ds.shuffle(len(y), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(HP["BATCH_SIZE"], drop_remainder=False).prefetch(AUTOTUNE)
# Map the CLASS_WEIGHT dictionary to a 1D array of weights for the training set
sw_tr = np.mean([[CLASS_WEIGHT[int(val)] for val in row] for row in y_tr], axis=1).astype(np.float32)
# Pass the sample weights only to the training dataset
ds_tr = make_ds(X_tr, M_tr, y_tr, r_tr, sw=sw_tr, shuffle=True)
ds_va = make_ds(X_va, M_va, y_va, r_va)
ds_te = make_ds(X_te, M_te, y_te, r_te)
steps_per_epoch = len(ds_tr)
log.info("Batches → train:%d  val:%d  test:%d",
         len(ds_tr), len(ds_va), len(ds_te))
# Label distribution → metrics/
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, (y_sp, title) in zip(axes,
        [(y_tr,"Train"),(y_va,"Val"),(y_te,"Test")]):
    vals, cnts = np.unique(y_sp, return_counts=True)
    ax.bar(vals, cnts, color="#4C72B0", edgecolor="white")
    ax.set_xticks(vals); ax.set_xticklabels([f"C{v}" for v in vals], fontsize=9)
    ax.set_title(f"{title} (n={len(y_sp):,})", fontweight="bold")
    ax.set_xlabel("Cell index"); ax.set_ylabel("Count")
plt.suptitle("MTL Label Distribution", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(str(PATHS["metrics"]/"mtl_label_dist.png"),
            dpi=150, bbox_inches="tight"); plt.close()


10:50:47 │ INFO     │ Class weights: {0: 0.872, 1: 1.013, 2: 1.014, 3: 1.017, 4: 0.996, 5: 0.996, 6: 1.026, 7: 1.031, 8: 1.027, 9: 1.031}


2026-05-30 10:50:48.022480: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-30 10:50:48.029093: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-30 10:50:48.031999: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

10:50:49 │ INFO     │ Batches → train:447  val:96  test:96


## Section 5 · Loss Functions

### Focal Loss (Head A — Classification)
```
FL = −α · (1 − p_t)^γ · log(p_t)
```
Down-weights Cell-0 (easy majority) and up-weights minority cells (hard).

### MSE Loss (Head B — Regression)  
```
L_MSE = mean( (rsrp_pred − rsrp_true)² )
```
Targets are standardised (mean≈0, std≈1), keeping MSE ≈ 1.0 at init,
comparable to Focal ≈ 2.0. The 0.5 weight prevents regression from 
drowning the classification gradient.

### Why MTL helps
The shared LSTM+Attention encoder must simultaneously satisfy:
- "Which cell has the best future signal?" (classification)
- "What exact dBm will that cell reach?" (regression)

The second objective forces the latent space to encode **signal trajectories**,
not just current snapshots — exactly what the Delta feature also provides.

In [10]:
# ─── Section 5 · Loss functions ──────────────────────────────────────────────

def focal_loss(gamma: float = 2.0, alpha: float = 0.25):
    """
    Multi-class Focal Loss for one-hot targets.
    p_t = sum(y_true * y_pred) — probability on correct class.
    weight = alpha * (1 - p_t)^gamma — down-weights confident examples.
    """
    def _focal(y_true, y_pred):
        y_pred  = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1-1e-7)
        y_true  = tf.cast(y_true, tf.float32)
        ce      = -y_true * tf.math.log(y_pred)              # (B, C)
        p_t     = tf.reduce_sum(y_true * y_pred,
                                axis=-1, keepdims=True)       # (B, 1)
        weight  = alpha * tf.pow(1.0 - p_t, gamma)           # (B, 1)
        return tf.reduce_sum(weight * ce, axis=-1)
    _focal.__name__ = f"focal_g{gamma}_a{alpha}"
    return _focal


# Instantiate with HP values
FOCAL_LOSS = focal_loss(HP["FOCAL_GAMMA"], HP["FOCAL_ALPHA"])
MSE_LOSS   = tf.keras.losses.MeanSquaredError()

log.info("Focal Loss: γ=%.1f  α=%.2f", HP["FOCAL_GAMMA"], HP["FOCAL_ALPHA"])
log.info("MSE on standardised RSRP (mean~0, std~1)")
log.info("Total: %.1f×Focal + %.1f×MSE",
         HP["LAMBDA_CLS"], HP["LAMBDA_REG"])

10:50:50 │ INFO     │ Focal Loss: γ=2.0  α=0.25
10:50:50 │ INFO     │ MSE on standardised RSRP (mean~0, std~1)
10:50:50 │ INFO     │ Total: 1.0×Focal + 0.5×MSE


## Section 6 · Custom Layers — Masked MHA & Set Transformer Block

In [11]:
# ─── Section 6 · Custom layers ───────────────────────────────────────────────

class MaskedMultiHeadAttention(keras.layers.Layer):
    """
    Multi-Head Self-Attention over MAX_CELLS with boolean key masking.
    Padded cells (mask=0) are invisible to all queries.
    """
    def __init__(self, num_heads, key_dim, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.mha = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=key_dim,
            dropout=dropout, dtype="float32")
        self._cfg = {"num_heads":num_heads,"key_dim":key_dim,"dropout":dropout}

    def call(self, x, mask, training=False):
        key_mask = tf.cast(mask, tf.bool)[:, tf.newaxis, tf.newaxis, :]
        return self.mha(query=x, value=x, key=x,
                        attention_mask=key_mask, training=training)

    def get_config(self):
        return {**super().get_config(), **self._cfg}


class SetTransformerBlock(keras.layers.Layer):
    """
    Pre-LN Set Transformer block over the cell-set axis.
    Residual connections + layer norm + masked MHA + GELU FFN.
    Padding cells are zeroed after each block to prevent gradient contamination.
    """
    def __init__(self, embed_dim, num_heads, key_dim, ff_dim,
                 dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self._cfg = dict(embed_dim=embed_dim, num_heads=num_heads,
                         key_dim=key_dim, ff_dim=ff_dim, dropout=dropout)
        self.norm1  = layers.LayerNormalization(epsilon=1e-6, dtype="float32")
        self.norm2  = layers.LayerNormalization(epsilon=1e-6, dtype="float32")
        self.mha    = MaskedMultiHeadAttention(num_heads, key_dim, dropout)
        self.ff1    = layers.Dense(ff_dim,    activation="gelu")
        self.ff2    = layers.Dense(embed_dim, activation=None)
        self.drop1  = layers.Dropout(dropout)
        self.drop2  = layers.Dropout(dropout)

    def call(self, x, mask, training=False):
        x = x + self.drop1(self.mha(self.norm1(x), mask, training), training)
        x = x + self.drop2(self.ff2(self.ff1(self.norm2(x))),       training)
        return x * tf.cast(mask[:, :, tf.newaxis], x.dtype)

    def get_config(self):
        return {**super().get_config(), **self._cfg}

print("MaskedMultiHeadAttention  defined.")
print("SetTransformerBlock       defined.")

MaskedMultiHeadAttention  defined.
SetTransformerBlock       defined.


## Section 7 · Multi-Task Set Transformer Model

### Architecture (two heads, one shared encoder)
```
Input (B, 10, 25, 6) + Mask (B, 10)
       │
  TimeDistributed(LSTM 64)      ← shared temporal encoder
       │                           same LSTM across all cells
  TimeDistributed(Dense 64)     ← Φ projection
       │
  N × SetTransformerBlock       ← pairwise cell attention (masked)
       │
  ┌────┴──────────────────────────────┐
  │                                   │
  │  Head A (Classification)          │  Head B (Regression)
  │  TimeDistributed Dense 64→1       │  GlobalAvgPool → Dense 32 → Dense 1
  │  Reshape → Masked Softmax         │  (float32 scalar per sample)
  │  (B, 10)  cls_output              │  () reg_output
  └───────────────────────────────────┘
```

### Why a separate regression head (not shared scoring)?
The regression head pools across ALL cells (GlobalAvg) to produce a single
RSRP estimate — it captures the *average quality* of the RF environment.
The classification head scores each cell *individually* via TimeDistributed.
Sharing the scoring layer would corrupt the regression gradient.

In [12]:
# ─── Section 7 · MTL Set Transformer Model ───────────────────────────────────
def build_mtl_transformer(hp: dict) -> keras.Model:
    C = hp["MAX_CELLS"]; W = hp["OBS_STEPS"]
    F = hp["N_FEATS"];   D = hp["PHI_DIM"]
    H = hp.get("TGT_STEPS", 5)
    # ── Inputs ────────────────────────────────────────────────────────────────
    inp_cells = keras.Input((C, W, F), name="cells",  dtype="float32")
    inp_mask  = keras.Input((C,),      name="mask",   dtype="float32")
    # ── Shared Stage 1: Temporal Encoder ─────────────────────────────────────
    # Shared LSTM weights across C cells captures universal RF trend grammar.
    trend = layers.TimeDistributed(
        layers.LSTM(hp["LSTM_UNITS"], return_sequences=False),
        name="td_lstm")(inp_cells)                              # (B, C, 64)
    # ── Shared Stage 2: Φ projection ─────────────────────────────────────────
    phi = layers.TimeDistributed(
        layers.Dense(D, activation="relu"), name="phi")(trend)
    phi = layers.TimeDistributed(
        layers.Dropout(hp["DROPOUT"]), name="phi_drop")(phi)   # (B, C, D)
    # ── Shared Stage 3: Set Transformer blocks ────────────────────────────────
    # Stacked MHA allows each cell to compare itself to ALL other cells.
    # After N_ST_BLOCKS, each cell's repr encodes pairwise ranking information.
    x = phi
    for i in range(hp["N_ST_BLOCKS"]):
        x = SetTransformerBlock(
            embed_dim=D, num_heads=hp["N_HEADS"],
            key_dim=hp["MHA_KEY_DIM"], ff_dim=hp["FF_DIM"],
            dropout=hp["DROPOUT"], name=f"st_block_{i}",
        )(x, inp_mask)                                          # (B, C, D)
    # ── Head A: Classification (Focal Loss) ───────────────────────────────────
    # Multi-horizon classification over H future timesteps
    rho = layers.TimeDistributed(
        layers.Dense(D, activation="relu"), name="rho")(x)
    rho = layers.TimeDistributed(
        layers.Dropout(hp["DROPOUT"]), name="rho_drop")(rho)
    # Dense outputs H future logits per cell: (B, C, H)
    logits = layers.TimeDistributed(
        layers.Dense(H, use_bias=True), name="scorer")(rho)
    # Permute to (B, H, C) so classification is performed across candidate cells per step
    logits = layers.Permute((2, 1), name="logits_permuted")(logits)  # (B, H, C)
    # Masked softmax: padding cells → −∞ → 0 probability
    cls_out = layers.Softmax(axis=-1, dtype="float32", name="cls_output")(
        layers.Add(name="pad_mask")([logits, layers.Reshape((1, C))((1.0 - inp_mask) * (-1e9))])
    )                                                            # (B, H, C)
    # ── Head B: Regression (MSE Loss) ────────────────────────────────────────
    # Pool across real cells only → predict expect future RSRP sequence (B, H)
    mask_exp = layers.Reshape((C, 1), name="mask_exp")(inp_mask)
    # Masked mean-pool: z = Σ x_i·mask_i / Σ mask_i
    x_pool   = tf.reduce_sum(x * tf.cast(mask_exp, x.dtype), axis=1)
    counts   = tf.maximum(tf.reduce_sum(mask_exp, axis=1), 1e-8)
    z_pool   = x_pool / tf.cast(counts, x_pool.dtype)           # (B, D)
    reg = layers.Dense(32, activation="relu",  name="reg_fc1")(z_pool)
    reg = layers.Dropout(hp["DROPOUT"],        name="reg_drop")(reg)
    # Predict RSRP sequence of shape (B, H)
    reg_out = layers.Dense(H, activation=None, dtype="float32",
                           name="reg_output")(reg)               # (B, H)
    model = keras.Model(
        inputs  = [inp_cells, inp_mask],
        outputs = {"cls_output": cls_out, "reg_output": reg_out},
        name    = "MTL_SetTransformer_HO",
    )
    return model
model = build_mtl_transformer(HP)
model.summary(line_length=90, expand_nested=False)
log.info("Parameters: %d", model.count_params())
log.info("Outputs: cls_output (B,10)  reg_output (B,)")


Model: "MTL_SetTransformer_HO"
__________________________________________________________________________________________
 Layer (type)              Output Shape               Param   Connected to                
                                                       #                                  
 cells (InputLayer)        [(None, 10, 25, 4)]        0       []                          
                                                                                          
 td_lstm (TimeDistributed  (None, 10, 64)             17664   ['cells[0][0]']             
 )                                                                                        
                                                                                          
 phi (TimeDistributed)     (None, 10, 64)             4160    ['td_lstm[0][0]']           
                                                                                          
 phi_drop (TimeDistribute  (None, 10, 64)             0    

## Section 8 · Compile — Weighted Multi-Task Loss

Keras `model.compile()` accepts a dict of losses (one per output name).
`loss_weights` multiplies each loss before summing:
```
L_total = 1.0 × L_focal(cls_output)  +  0.5 × L_mse(reg_output)
```

Metrics:
- `cls_output` → `CategoricalAccuracy` (top-1) and `TopKCategoricalAccuracy` (top-3)
- `reg_output` → `MeanAbsoluteError` as a diagnostic for signal-prediction quality

In [13]:
# ─── Section 8 · LR schedule + compile ───────────────────────────────────────

class WarmUpCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, lr_max, lr_min, warmup_steps, decay_steps):
        super().__init__()
        self.lr_max=float(lr_max); self.lr_min=float(lr_min)
        self.warmup_steps=float(warmup_steps)
        self.decay_steps=float(decay_steps)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        warmup = self.lr_max * step / tf.maximum(self.warmup_steps, 1.0)
        cosine = self.lr_min + 0.5*(self.lr_max-self.lr_min)*(
            1.0 + tf.cos(np.pi *
                tf.minimum(step-self.warmup_steps, self.decay_steps)
                / self.decay_steps))
        return tf.where(step < self.warmup_steps, warmup, cosine)

    def get_config(self):
        return {"lr_max":self.lr_max,"lr_min":self.lr_min,
                "warmup_steps":self.warmup_steps,"decay_steps":self.decay_steps}


warmup_steps = HP["LR_WARMUP_EP"] * steps_per_epoch
decay_steps  = HP["LR_DECAY_EP"]  * steps_per_epoch
lr_sched     = WarmUpCosineDecay(HP["LR_INIT"], HP["LR_INIT"]*0.01,
                                  warmup_steps, decay_steps)

model.compile(
    optimizer    = keras.optimizers.Adam(learning_rate=lr_sched),

    # ── Per-output loss functions ─────────────────────────────────────────────
    loss         = {
        "cls_output": FOCAL_LOSS,
        "reg_output": MSE_LOSS,
    },
    # ── Weighted combination: L_total = 1.0×Focal + 0.5×MSE ─────────────────
    loss_weights = {
        "cls_output": HP["LAMBDA_CLS"],   # 1.0
        "reg_output": HP["LAMBDA_REG"],   # 0.5
    },
    # ── Per-output metrics ────────────────────────────────────────────────────
    metrics = {
        "cls_output": [
            keras.metrics.CategoricalAccuracy(name="top1_acc"),
            keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
        ],
        "reg_output": [
            keras.metrics.MeanAbsoluteError(name="rsrp_mae"),
        ],
    },
)
log.info("Compiled. Total loss = %.1f×Focal + %.1f×MSE",
         HP["LAMBDA_CLS"], HP["LAMBDA_REG"])

10:50:53 │ INFO     │ Compiled. Total loss = 1.0×Focal + 0.5×MSE


## Section 9 · Callbacks

In [14]:
# ─── Section 9 · Callbacks ───────────────────────────────────────────────────
#
# Monitor = val_cls_output_top1_acc
# Keras auto-generates metric names as <output_name>_<metric_name>

MONITOR = "val_cls_output_top1_acc"

class MLflowEpochCB(keras.callbacks.Callback):
    """Log all epoch metrics to MLflow; no-ops if server is unreachable."""
    def on_epoch_end(self, epoch, logs=None):
        if MLFLOW_OK and logs:
            for k, v in logs.items():
                mlflow.log_metric(k, float(v), step=epoch)

if MLFLOW_OK:
    _run = mlflow.start_run(run_name="mtl_transformer")
    mlflow.log_params(HP)
    mlflow.log_param("loss_fn", f"{HP['LAMBDA_CLS']}×Focal+{HP['LAMBDA_REG']}×MSE")
    _rid = _run.info.run_id
    log.info("MLflow run: %s", _rid)
else:
    _rid = None

callbacks = [
    # ── Monitor Top-1 directly (targets breaking 57% ceiling) ────────────────
    keras.callbacks.EarlyStopping(
        monitor=MONITOR, patience=12, min_delta=1e-4,
        restore_best_weights=True, mode="max", verbose=1),
    keras.callbacks.ModelCheckpoint(
        filepath=CKPT_PATH, monitor=MONITOR,
        save_best_only=True, mode="max", verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor=MONITOR, factor=0.5, patience=6,
        min_lr=1e-7, mode="max", verbose=1),
    keras.callbacks.TensorBoard(
        log_dir=str(PATHS["tb_logs"]),
        histogram_freq=0, write_graph=True, update_freq="epoch"),
    keras.callbacks.CSVLogger(
        str(PATHS["metrics"]/"training_log.csv"), append=False),
    MLflowEpochCB(),
]
log.info("Callbacks ready. Monitor='%s'  CKPT='%s'", MONITOR, CKPT_PATH)

10:50:54 │ INFO     │ MLflow run: 63c896222a06418598ed6659f30148ef
10:50:54 │ INFO     │ Callbacks ready. Monitor='val_cls_output_top1_acc'  CKPT='/home/wassimmchichi/Downloads/Handover_projects/models/best_mtl_transformer.keras'


## Section 10 · Training

In [15]:
# ─── Section 10 · Training ───────────────────────────────────────────────────

log.info("Training — %d train | %d val | batch=%d | max_ep=%d",
         len(y_tr), len(y_va), HP["BATCH_SIZE"], HP["EPOCHS"])

history = model.fit(
    ds_tr,
    validation_data = ds_va,
    epochs          = HP["EPOCHS"],
    callbacks       = callbacks,
    verbose         = 1,
    # REMOVED class_weight argument completely
)

_hist = history.history
best_ep    = int(np.argmax(_hist[MONITOR])) + 1
best_top1  = float(max(_hist[MONITOR]))
best_mae   = float(_hist["val_reg_output_mae"][best_ep-1]) if "val_reg_output_mae" in _hist else 0.0

log.info("Done — best %s=%.4f @ epoch %d  |  RSRP_MAE=%.4f",
         MONITOR, best_top1, best_ep, best_mae)

if MLFLOW_OK:
    mlflow.log_metrics({"best_val_top1.keras": best_top1,
                         "best_val_rsrp_mae.keras": best_mae,
                         "best_epoch.keras": float(best_ep)})

10:50:54 │ INFO     │ Training — 57120 train | 12240 val | batch=128 | max_ep=60
Epoch 1/60


2026-05-30 10:51:02.769466: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2026-05-30 10:51:04.964506: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f509275a7b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-05-30 10:51:04.964539: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 2060, Compute Capability 7.5
2026-05-30 10:51:04.971959: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1780134665.080292   41262 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  4/447 [..............................] - ETA: 9s - loss: 1.3792 - cls_output_loss: 0.4361 - reg_output_loss: 1.8862 - cls_output_top1_acc: 0.1898 - cls_output_top3_acc: 0.4508 - reg_output_rsrp_mae: 1.0948     WARNING:tensorflow:Callback method `on_train_batch_end` is slow compared to the batch time (batch time: 0.0187s vs `on_train_batch_end` time: 0.0245s). Check your callbacks.
10:51:09 │ WARNING  │ Callback method `on_train_batch_end` is slow compared to the batch time (batch time: 0.0187s vs `on_train_batch_end` time: 0.0245s). Check your callbacks.
445/447 [============================>.] - ETA: 0s - loss: 0.6613 - cls_output_loss: 0.3168 - reg_output_loss: 0.6889 - cls_output_top1_acc: 0.3569 - cls_output_top3_acc: 0.6764 - reg_output_rsrp_mae: 0.6159
Epoch 1: val_cls_output_top1_acc improved from -inf to 0.51356, saving model to /home/wassimmchichi/Downloads/Handover_projects/models/best_mtl_transformer.keras
447/447 [==============================] - 27s 27ms/step - loss: 0.

In [16]:
_out = PATHS["metrics"] / "mtl-transformer-architecture.png"

tf.keras.utils.plot_model(
    model,
    to_file=str(_out),
    show_shapes=True,
    show_layer_names=True,
    dpi=150
)

log.info("Saved: %s", _out)

if MLFLOW_OK:
    mlflow.log_artifact(str(_out))

10:56:09 │ INFO     │ Saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/mtl_transformer/mtl-transformer-architecture.png


## Section 11 · Training Curves  →  `../../metrics/`

In [17]:
# ─── Section 11 · Training curves ────────────────────────────────────────────

hist = history.history
ep   = range(1, len(hist["loss"]) + 1)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (tr_k, va_k, title, hi) in zip(axes, [
    ("loss",                        "val_loss",
     "Total Loss",                   False),
    ("cls_output_top1_acc",         "val_cls_output_top1_acc",
     "Top-1 Acc (primary KPI)",     True),
    ("cls_output_top3_acc",         "val_cls_output_top3_acc",
     "Top-3 Accuracy",               True),
    ("reg_output_rsrp_mae",         "val_reg_output_rsrp_mae",
     "RSRP MAE (diagnostic)",        False),
]):
    if tr_k not in hist: continue
    ax.plot(ep, hist[tr_k], lw=2, label="train")
    ax.plot(ep, hist[va_k], lw=2, ls="--", label="val")
    fn  = np.argmax if hi else np.argmin
    be  = fn(hist[va_k]) + 1
    bv  = (max if hi else min)(hist[va_k])
    ax.axvline(be, color="red", ls=":", lw=1.2, alpha=0.8)
    ax.scatter([be], [bv], color="red", zorder=5, s=70,
               label=f"best @ ep{be} ({bv:.4f})")
    ax.set(title=title, xlabel="Epoch")
    ax.legend(fontsize=7); ax.grid(alpha=0.4)

fig.suptitle("MTL Set Transformer — Experiment 3", fontsize=13, fontweight="bold")
plt.tight_layout()
_out = PATHS["metrics"] / "mtl_training_curves.png"
plt.savefig(str(_out), dpi=150, bbox_inches="tight"); plt.close()
log.info("Saved: %s", _out)
if MLFLOW_OK: mlflow.log_artifact(str(_out))

10:56:12 │ INFO     │ Saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/mtl_transformer/mtl_training_curves.png


## Section 12 · Model Evaluation — Load `best_mtl_transformer.keras`

In [18]:
# ─── Section 12 · Evaluation ─────────────────────────────────────────────────
#
# Reload from disk to verify the saved checkpoint is complete and
# identical to the in-memory model (smoke test for the MLOps pipeline).
_BEST = CKPT_PATH
log.info("Loading: %s", _BEST)
model = keras.models.load_model(
    _BEST,
    compile=False,
    custom_objects={
        "MaskedMultiHeadAttention": MaskedMultiHeadAttention,
        "SetTransformerBlock"     : SetTransformerBlock,
        "WarmUpCosineDecay"       : WarmUpCosineDecay,
    FOCAL_LOSS.__name__: FOCAL_LOSS},
)
preds_te  = model.predict(ds_te, verbose=1)
probs_te  = preds_te["cls_output"]    # (N, H, 10)
rsrp_pred = preds_te["reg_output"]    # (N, H)
y_pred_te = probs_te.argmax(axis=-1)  # (N, H)
# Flatten sequence dimensions for global evaluation across all timesteps
y_te_flat = y_te.ravel()
y_pred_te_flat = y_pred_te.ravel()
probs_te_flat = probs_te.reshape(-1, probs_te.shape[-1])
# ── Classification metrics ────────────────────────────────────────────────────
top1 = float((y_pred_te_flat == y_te_flat).mean())
top3 = float(top_k_accuracy_score(y_te_flat, probs_te_flat, k=3, labels=ALL_LABELS))
top5 = float(top_k_accuracy_score(y_te_flat, probs_te_flat, k=5, labels=ALL_LABELS))
# ── Regression metrics (convert back to dBm) ─────────────────────────────────
if 'scaler_r' not in globals(): scaler_r = _data["scalers"]["r"]
rsrp_pred_dbm = scaler_r.inverse_transform(rsrp_pred.reshape(-1,1)).reshape(rsrp_pred.shape)
rsrp_true_dbm = scaler_r.inverse_transform(r_te.reshape(-1,1)).reshape(r_te.shape)
rsrp_mae_dbm  = float(mean_absolute_error(rsrp_true_dbm.ravel(), rsrp_pred_dbm.ravel()))
log.info("Test → Top-1:%.4f  Top-3:%.4f  Top-5:%.4f  RSRP_MAE:%.2f dBm",
         top1, top3, top5, rsrp_mae_dbm)
if MLFLOW_OK:
    mlflow.log_metrics({"test_top1": top1, "test_top3": top3,
                         "test_top5": top5,
                         "test_rsrp_mae_dbm": rsrp_mae_dbm})
print("=" * 65)
print("  EXPERIMENT 3 TEST RESULTS")
print("=" * 65)
print(f"  Top-1 Accuracy  : {top1:.4f}   ({top1*100:.2f}%)")
print(f"  Top-3 Accuracy  : {top3:.4f}   ({top3*100:.2f}%)")
print(f"  Top-5 Accuracy  : {top5:.4f}   ({top5*100:.2f}%)")
print(f"  RSRP MAE        : {rsrp_mae_dbm:.2f} dBm  (regression diagnostic)")
print()
print(classification_report(
    y_te_flat, y_pred_te_flat,
    labels=ALL_LABELS,
    target_names=[f"Cell {i}" for i in range(HP["MAX_CELLS"])],
    digits=4, zero_division=0,
))


10:56:12 │ INFO     │ Loading: /home/wassimmchichi/Downloads/Handover_projects/models/best_mtl_transformer.keras
96/96 [==============================] - 1s 6ms/step
10:56:15 │ INFO     │ Test → Top-1:0.5579  Top-3:0.8602  Top-5:0.9499  RSRP_MAE:4.20 dBm
  EXPERIMENT 3 TEST RESULTS
  Top-1 Accuracy  : 0.5579   (55.79%)
  Top-3 Accuracy  : 0.8602   (86.02%)
  Top-5 Accuracy  : 0.9499   (94.99%)
  RSRP MAE        : 4.20 dBm  (regression diagnostic)

              precision    recall  f1-score   support

      Cell 0     0.5895    0.4970    0.5393      7189
      Cell 1     0.5402    0.5382    0.5392      6048
      Cell 2     0.5637    0.5877    0.5754      6032
      Cell 3     0.5549    0.5554    0.5552      6048
      Cell 4     0.5548    0.5796    0.5669      5927
      Cell 5     0.5455    0.5614    0.5533      5798
      Cell 6     0.5679    0.5864    0.5770      6085
      Cell 7     0.5575    0.5746    0.5660      6063
      Cell 8     0.5517    0.5632    0.5574      6127
      C

## Section 13 · Confusion Matrix & Per-Cell Recall  →  `../../metrics/`

In [24]:
# ─── Section 13 · Confusion matrix ───────────────────────────────────────────
C  = HP["MAX_CELLS"]
cm = confusion_matrix(y_te_flat, y_pred_te_flat, labels=list(range(C)))
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
cl = [f"C{i}" for i in range(C)]
errors = (rsrp_pred_dbm - rsrp_true_dbm).ravel()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[f"P-{l}" for l in cl],
            yticklabels=[f"T-{l}" for l in cl],
            linewidths=0.5, ax=axes[0], annot_kws={"size":9})
axes[0].set(title="Confusion Matrix — Counts",
            ylabel="Actual", xlabel="Predicted")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="YlGn",
            xticklabels=[f"P-{l}" for l in cl],
            yticklabels=[f"T-{l}" for l in cl],
            linewidths=0.5, ax=axes[1], vmin=0, vmax=1,
            annot_kws={"size":9})
axes[1].set(title="Normalised — Recall per Row",
            ylabel="Actual", xlabel="Predicted")
plt.tight_layout()
_out = PATHS["metrics"] / "confusion_matrix.png"
plt.savefig(str(_out), dpi=150, bbox_inches="tight"); plt.close()
if MLFLOW_OK: mlflow.log_artifact(str(_out))
per_recall = cm_norm.diagonal()
fig, ax = plt.subplots(figsize=(9, 3.5))
bars = ax.bar(range(C), per_recall,
              color=["#2196F3" if v>=0.5 else "#F44336" for v in per_recall],
              edgecolor="white")
ax.axhline(top1, color="black", ls="--", lw=1.2,
           label=f"Overall Top-1 ({top1:.3f})")
for b,v in zip(bars, per_recall):
    ax.text(b.get_x()+b.get_width()/2, v+0.015, f"{v:.2f}",
            ha="center", va="bottom", fontsize=8)
ax.set(xticks=range(C), xticklabels=[f"Cell {i}" for i in range(C)],
       ylabel="Recall", ylim=(0,1.18), title="Per-Cell Recall — MTL Set Transformer")
ax.legend(); ax.grid(axis="y", alpha=0.4)
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
_out = PATHS["metrics"] / "per_cell_recall.png"
plt.savefig(str(_out), dpi=150, bbox_inches="tight"); plt.close()
if MLFLOW_OK: mlflow.log_artifact(str(_out))
log.info("Confusion matrix + per-cell recall saved.")


11:04:41 │ INFO     │ Confusion matrix + per-cell recall saved.


## Section 14 · Regression Diagnostic — RSRP Prediction Quality

In [26]:
# ─── Section 14 · Regression diagnostic ─────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: predicted vs true
axes[0].scatter(
    rsrp_true_dbm.ravel()[:5000],
    rsrp_pred_dbm.ravel()[:5000],
    alpha=0.3,
    s=6
)
_lo = min(rsrp_true_dbm.min(), rsrp_pred_dbm.min())
_hi = max(rsrp_true_dbm.max(), rsrp_pred_dbm.max())
axes[0].plot([_lo,_hi],[_lo,_hi],"r--",lw=1.2,label="Perfect prediction")
axes[0].set(xlabel="True RSRP (dBm)", ylabel="Predicted RSRP (dBm)",
            title=f"Head B Regression  (MAE={rsrp_mae_dbm:.2f} dBm)")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.4)

# Error distribution
errors = (rsrp_pred_dbm - rsrp_true_dbm).ravel()
axes[1].hist(errors, bins=60, color="#DD8452", edgecolor="white", linewidth=0.5)
axes[1].axvline(0, color="red", lw=1.5, label="Zero error")
axes[1].axvline(errors.mean(), color="blue", lw=1.2, ls="--",
                label=f"Mean error {errors.mean():.2f} dBm")
axes[1].set(xlabel="Prediction error (dBm)", ylabel="Count",
            title="RSRP Prediction Error Distribution")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.4)

plt.suptitle("Head B (Regression) — RSRP Prediction Diagnostic",
             fontsize=12, fontweight="bold")
plt.tight_layout()
_out = PATHS["metrics"] / "rsrp_regression_diagnostic.png"
plt.savefig(str(_out), dpi=150, bbox_inches="tight"); plt.close()
log.info("Saved: %s", _out)
if MLFLOW_OK: mlflow.log_artifact(str(_out))

11:05:36 │ INFO     │ Saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/mtl_transformer/rsrp_regression_diagnostic.png


In [27]:
print(type(errors))
print(errors.shape)

<class 'numpy.ndarray'>
(61200,)


## Section 15 · Experiment Comparison Plot  →  `../../metrics/exp3_results.png`

In [28]:
# ─── Section 15 · Experiment comparison ──────────────────────────────────────
# Update BASELINE_SCORES with actual values from notebooks 01 and 02.

BASELINE_SCORES = {
    "DeepSet (NB01)":       {"top1": 0.53, "top3": 0.78, "top5": 0.91},
    "SetTransformer (NB02)":{"top1": 0.57, "top3": 0.82, "top5": 0.93},
}
MTL_SCORES = {"top1": top1, "top3": top3, "top5": top5}

_models = list(BASELINE_SCORES.keys()) + ["MTL SetTransformer (Exp3)"]
_top1   = [v["top1"] for v in BASELINE_SCORES.values()] + [MTL_SCORES["top1"]]
_top3   = [v["top3"] for v in BASELINE_SCORES.values()] + [MTL_SCORES["top3"]]
_colors = ["#4C72B0","#55A868","#DD8452"]

x = np.arange(len(_models)); w = 0.35
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grouped bar: Top-1 vs Top-3
for i,(m,c,t1,t3) in enumerate(zip(_models,_colors,_top1,_top3)):
    axes[0].bar(i-w/2, t1, w, color=c, alpha=0.85, edgecolor="white",
                label=f"{m[:20]} (T1={t1:.3f})")
    axes[0].bar(i+w/2, t3, w, color=c, alpha=0.45, edgecolor="white",
                hatch="///", label=f"Top-3={t3:.3f}")
    axes[0].text(i-w/2, t1+0.005, f"{t1:.3f}", ha="center", fontsize=8, fontweight="bold")
    axes[0].text(i+w/2, t3+0.005, f"{t3:.3f}", ha="center", fontsize=8)
axes[0].axhline(0.57, color="red", ls="--", lw=1.2,
                label="57% ceiling (Exp2 baseline)")
axes[0].set(xticks=x, xticklabels=[m.split("(")[0].strip() for m in _models],
            ylabel="Accuracy", ylim=(0,1.1),
            title="Top-1 vs Top-3 — All Experiments")
axes[0].legend(fontsize=7, loc="lower right"); axes[0].grid(axis="y", alpha=0.4)

# Delta bar: improvement over DeepSet baseline
base_t1 = BASELINE_SCORES["DeepSet (NB01)"]["top1"]
deltas  = [v["top1"]-base_t1 for v in BASELINE_SCORES.values()]
deltas += [MTL_SCORES["top1"]-base_t1]
d_colors= ["#F44336" if d<0 else "#4CAF50" for d in deltas]
axes[1].bar(x, deltas, color=d_colors, edgecolor="white")
axes[1].axhline(0, color="black", lw=1)
for i,d in enumerate(deltas):
    axes[1].text(i, d+(0.003 if d>=0 else -0.008), f"{d:+.3f}",
                 ha="center", fontsize=9, fontweight="bold")
axes[1].set(xticks=x, xticklabels=[m.split("(")[0].strip() for m in _models],
            ylabel="Δ Top-1 vs DeepSet baseline",
            title="Improvement from DeepSet Baseline")
axes[1].grid(axis="y", alpha=0.4)

plt.suptitle("Experiment 3 — MTL Set Transformer Results",
             fontsize=13, fontweight="bold")
plt.tight_layout()
_out = PATHS["metrics"] / "exp3_results.png"
plt.savefig(str(_out), dpi=150, bbox_inches="tight"); plt.close()
log.info("Saved: %s", _out)
if MLFLOW_OK: mlflow.log_artifact(str(_out))

11:05:39 │ INFO     │ Saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/mtl_transformer/exp3_results.png


## Section 16 · Metadata, Final Model & MLflow Close

In [29]:
# ─── Section 16 · Save artefacts ─────────────────────────────────────────────

model.save(FINAL_PATH)
log.info("Final model: %s", FINAL_PATH)

meta = {
    "experiment"      : "Experiment 3 — MTL + Delta/Margin Features",
    "created"         : datetime.datetime.now().isoformat(),
    "notebook"        : "notebooks/modeling/03_mtl_transformer.ipynb",
    "best_checkpoint.keras" : CKPT_PATH,
    "final_model"     : FINAL_PATH,
    "test_top1_acc"   : round(top1, 4),
    "test_top3_acc"   : round(top3, 4),
    "test_top5_acc"   : round(top5, 4),
    "test_rsrp_mae_dbm": round(rsrp_mae_dbm, 3),
    "best_epoch.keras"      : best_ep,
    "hyperparams"     : HP,
    "feature_names"   : F_NAMES,
    "loss"            : f"{HP['LAMBDA_CLS']}×Focal(γ={HP['FOCAL_GAMMA']},α={HP['FOCAL_ALPHA']}) "
                        f"+ {HP['LAMBDA_REG']}×MSE",
    "previous_ceiling": 0.57,
    "improvement"     : round(top1 - 0.57, 4),
    "paths"           : {k: str(v) for k, v in PATHS.items()},
}
_meta_out = PATHS["metrics"] / "transformer_metadata.json"
json.dump(meta, open(str(_meta_out),"w"), indent=2)
log.info("Metadata: %s", _meta_out)

if MLFLOW_OK:
    for p in PATHS["metrics"].glob("*.png"):
        mlflow.log_artifact(str(p))
    mlflow.log_artifact(str(_meta_out))
    mlflow.tensorflow.log_model(
        model,
        artifact_path="mtl_transformer_keras",
        registered_model_name="handover_mtl_transformer",
    )
    mlflow.end_run()
    log.info("MLflow run closed.")

print()
print("=" * 65)
print("  ARTEFACT INVENTORY")
print("=" * 65)
for lbl, d in [("models/",  PATHS["models"]),
               ("metrics/", PATHS["metrics"])]:
    print(f"\n  {lbl}")
    for p in sorted(Path(d).iterdir()):
        if p.is_file():
            print(f"    {p.name:<44s} {p.stat().st_size/1024:>7.1f} KB")

print()
print(f"  Top-1 : {top1:.4f}  (prev ceiling: 0.5700  Δ={top1-0.57:+.4f})")
print(f"  Top-3 : {top3:.4f}")
print(f"  RSRP MAE: {rsrp_mae_dbm:.2f} dBm")
print()
print("  TensorBoard:  tensorboard --logdir ../../tb_logs/")
print("  MLflow UI:    mlflow ui --backend-store-uri file://$(pwd)/../../mlflow/mlruns")
print("  EC2 tunnel:   ssh -N -L 5000:localhost:5000 -i key.pem ec2-user@<EC2_IP>")

11:05:40 │ INFO     │ Final model: /home/wassimmchichi/Downloads/Handover_projects/models/mtl_transformer_final.keras
11:05:40 │ INFO     │ Metadata: /home/wassimmchichi/Downloads/Handover_projects/metrics/mtl_transformer/transformer_metadata.json


2026/05/30 11:05:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 11:05:51 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


11:05:51 │ WARNING  │ Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.
INFO:tensorflow:Assets written to: /tmp/tmpk2bxynvb/model/data/model/assets
11:05:59 │ INFO     │ Assets written to: /tmp/tmpk2bxynvb/model/data/model/assets


Registered model 'handover_mtl_transformer' already exists. Creating a new version of this model...
2026/05/30 11:06:26 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: handover_mtl_transformer, version 2
Created version '2' of model 'handover_mtl_transformer'.


🏃 View run mtl_transformer at: http://127.0.0.1:5000/#/experiments/15/runs/63c896222a06418598ed6659f30148ef
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/15
11:06:26 │ INFO     │ MLflow run closed.

  ARTEFACT INVENTORY

  models/
    6g_predictive_final.keras                     1541.6 KB
    best_6g_predictive.keras                      7212.0 KB
    best_honet_final.keras                        3281.3 KB
    best_honet_p1.keras                           1545.1 KB
    best_honet_p2.keras                           2516.3 KB
    best_mh_transformer.keras                     3407.2 KB
    best_mtl_transformer.keras                    2096.9 KB
    best_set_transformer.keras                    2650.9 KB
    best_st_deepset.keras                         9277.2 KB
    best_strategic_deepset.keras                  2404.2 KB
    best_temporal_deepset.keras                    764.1 KB
    cell_scaler.pkl                                  0.6 KB
    mtl_transformer_final.keras      